# 02. Data Cleaning and Transformation

## Objective

This notebook converts the validated Olist raw data into analysis-ready tables.

**DuckDB is the main data-processing layer in this notebook.** The raw CSV
files are read directly by DuckDB, cleaning rules are applied in SQL, and
the cleaned tables are stored in the persistent DuckDB database.

Pandas is used only for displaying results and exporting final CSV copies
where convenient.

The raw CSV files are not modified.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
RAW_DIR = PROJECT_ROOT / "02_Data" / "01_Raw"
DUCKDB_DIR = PROJECT_ROOT / "04_DuckDB"
PROCESSED_DIR = PROJECT_ROOT / "02_Data" / "02_Processed"
ANALYTICAL_DIR = PROJECT_ROOT / "02_Data" / "03_Analytical"

DUCKDB_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
ANALYTICAL_DIR.mkdir(parents=True, exist_ok=True)

DUCKDB_PATH = DUCKDB_DIR / "olist_capstone.duckdb"
con = duckdb.connect(str(DUCKDB_PATH))

print(f"DuckDB version: {duckdb.__version__}")
print(f"Database: {DUCKDB_PATH}")
print(f"Raw directory exists: {RAW_DIR.exists()}")

DuckDB version: 1.5.5
Database: d:\Learning\Data Science\Capstone Project\Final_Project_Olist_BYOC_Capstone\04_DuckDB\olist_capstone.duckdb
Raw directory exists: True


## 1. Load Raw Data into DuckDB

The nine raw CSV files are registered as DuckDB views.

The cleaning work therefore starts from the raw files inside DuckDB rather
than loading the raw data into Pandas first.

In [ ]:
raw_files = {
    "raw_customers": RAW_DIR / "olist_customers_dataset.csv",
    "raw_geolocation": RAW_DIR / "olist_geolocation_dataset.csv",
    "raw_orders": RAW_DIR / "olist_orders_dataset.csv",
    "raw_order_items": RAW_DIR / "olist_order_items_dataset.csv",
    "raw_payments": RAW_DIR / "olist_order_payments_dataset.csv",
    "raw_reviews": RAW_DIR / "olist_order_reviews_dataset.csv",
    "raw_products": RAW_DIR / "olist_products_dataset.csv",
    "raw_sellers": RAW_DIR / "olist_sellers_dataset.csv",
    "raw_category_translation": RAW_DIR / "product_category_name_translation.csv",
}

for view_name, file_path in raw_files.items():
    if not file_path.exists():
        raise FileNotFoundError(f"Missing raw file: {file_path}")
    path_sql = str(file_path.resolve()).replace("'", "''")
    con.execute(
        f"""CREATE OR REPLACE VIEW "{view_name}" AS
        SELECT * FROM read_csv_auto('{path_sql}', header=True)"""
    )

display(con.sql("""
SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM raw_customers
UNION ALL SELECT 'geolocation', COUNT(*) FROM raw_geolocation
UNION ALL SELECT 'orders', COUNT(*) FROM raw_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM raw_order_items
UNION ALL SELECT 'payments', COUNT(*) FROM raw_payments
UNION ALL SELECT 'reviews', COUNT(*) FROM raw_reviews
UNION ALL SELECT 'products', COUNT(*) FROM raw_products
UNION ALL SELECT 'sellers', COUNT(*) FROM raw_sellers
UNION ALL SELECT 'category_translation', COUNT(*) FROM raw_category_translation
ORDER BY table_name
""").df())

,table_name,row_count
0,category_translation,71
1,customers,99441
2,geolocation,1000163
3,order_items,112650
4,orders,99441
5,payments,103886
6,products,32951
7,reviews,99224
8,sellers,3095


## 2. Data Clean - Orders

All order records are retained.

Timestamp fields are converted using `TRY_CAST`. Delivery duration and
delivery delay are calculated in SQL. Missing actual delivery dates remain
missing because they represent orders where the delivery event is not
available.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE orders_clean AS
WITH typed AS (
    SELECT
        order_id, customer_id, order_status,
        TRY_CAST(order_purchase_timestamp AS TIMESTAMP) AS order_purchase_timestamp,
        TRY_CAST(order_approved_at AS TIMESTAMP) AS order_approved_at,
        TRY_CAST(order_delivered_carrier_date AS TIMESTAMP) AS order_delivered_carrier_date,
        TRY_CAST(order_delivered_customer_date AS TIMESTAMP) AS order_delivered_customer_date,
        TRY_CAST(order_estimated_delivery_date AS TIMESTAMP) AS order_estimated_delivery_date
    FROM raw_orders
)
SELECT *,
    CASE WHEN order_delivered_customer_date IS NOT NULL
          AND order_purchase_timestamp IS NOT NULL
         THEN EXTRACT(EPOCH FROM (order_delivered_customer_date - order_purchase_timestamp))/86400.0
    END AS delivery_duration_days,
    CASE WHEN order_estimated_delivery_date IS NOT NULL
          AND order_purchase_timestamp IS NOT NULL
         THEN EXTRACT(EPOCH FROM (order_estimated_delivery_date - order_purchase_timestamp))/86400.0
    END AS estimated_delivery_days,
    CASE WHEN order_delivered_customer_date IS NOT NULL
          AND order_estimated_delivery_date IS NOT NULL
         THEN EXTRACT(EPOCH FROM (order_delivered_customer_date - order_estimated_delivery_date))/86400.0
    END AS delivery_delay_days,
    CASE
        WHEN order_delivered_customer_date IS NULL
          OR order_estimated_delivery_date IS NULL THEN NULL
        WHEN order_delivered_customer_date > order_estimated_delivery_date THEN 1
        ELSE 0
    END AS is_late
FROM typed
""")

display(con.sql("""
SELECT
    COUNT(*) AS orders_retained,
    COUNT(order_delivered_customer_date) AS valid_actual_delivery_date,
    COUNT(delivery_delay_days) AS valid_delivery_delay,
    SUM(CASE WHEN is_late = 1 THEN 1 ELSE 0 END) AS late_orders,
    SUM(CASE WHEN is_late = 0 THEN 1 ELSE 0 END) AS on_time_or_early,
    SUM(CASE WHEN is_late IS NULL THEN 1 ELSE 0 END) AS unclassified
FROM orders_clean
""").df())

,orders_retained,valid_actual_delivery_date,valid_delivery_delay,late_orders,on_time_or_early,unclassified
0,99441,96476,96476,7827.0,88649.0,2965.0


## 3. Data Clean - Order Items

The native order-item grain is preserved.

Numeric fields are converted using `TRY_CAST`. No row deletion or
imputation is applied because Notebook 01 showed no missing or negative
values in the relevant numeric fields.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE order_items_clean AS
SELECT
    order_id,
    TRY_CAST(order_item_id AS INTEGER) AS order_item_id,
    product_id, seller_id,
    TRY_CAST(shipping_limit_date AS TIMESTAMP) AS shipping_limit_date,
    TRY_CAST(price AS DOUBLE) AS price,
    TRY_CAST(freight_value AS DOUBLE) AS freight_value
FROM raw_order_items
""")

display(con.sql("""
SELECT
    COUNT(*) AS rows_retained,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN price IS NULL THEN 1 ELSE 0 END) AS missing_price,
    SUM(CASE WHEN freight_value IS NULL THEN 1 ELSE 0 END) AS missing_freight,
    SUM(CASE WHEN price < 0 THEN 1 ELSE 0 END) AS negative_price,
    SUM(CASE WHEN freight_value < 0 THEN 1 ELSE 0 END) AS negative_freight
FROM order_items_clean
""").df())

display(con.sql("""
SELECT
    COUNT(*) AS order_item_rows,
    SUM(CASE WHEN oi.shipping_limit_date IS NULL THEN 1 ELSE 0 END)
        AS missing_shipping_limit_dates,
    SUM(CASE WHEN oi.shipping_limit_date < o.order_purchase_timestamp THEN 1 ELSE 0 END)
        AS shipping_limit_before_purchase
FROM order_items_clean oi
LEFT JOIN orders_clean o USING (order_id)
""").df())

,rows_retained,unique_orders,missing_price,missing_freight,negative_price,negative_freight
0,112650,98666,0.0,0.0,0.0,0.0


,order_item_rows,missing_shipping_limit_dates,shipping_limit_before_purchase
0,112650,0.0,0.0


## 4. Data Clean - Payments

Payment records remain at their native grain.

Zero payment values, zero installment values and observed payment sequence
values are retained because Notebook 01 identified them as source-derived
values rather than errors that can be safely corrected.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE payments_clean AS
SELECT
    order_id,
    TRY_CAST(payment_sequential AS INTEGER) AS payment_sequential,
    payment_type,
    TRY_CAST(payment_installments AS INTEGER) AS payment_installments,
    TRY_CAST(payment_value AS DOUBLE) AS payment_value
FROM raw_payments
""")

display(con.sql("""
SELECT
    COUNT(*) AS payment_rows,
    COUNT(DISTINCT order_id) AS unique_orders,
    SUM(CASE WHEN payment_value <= 0 THEN 1 ELSE 0 END) AS payment_value_non_positive,
    SUM(CASE WHEN payment_installments <= 0 THEN 1 ELSE 0 END) AS installments_non_positive,
    SUM(CASE WHEN payment_sequential <= 0 THEN 1 ELSE 0 END) AS sequence_non_positive
FROM payments_clean
""").df())

display(con.sql("""
WITH s AS (
    SELECT order_id, COUNT(*) payment_count,
           COUNT(DISTINCT payment_sequential) unique_sequences,
           MIN(payment_sequential) min_sequence,
           MAX(payment_sequential) max_sequence
    FROM payments_clean
    GROUP BY order_id
)
SELECT COUNT(*) AS orders_represented,
       SUM(CASE WHEN unique_sequences < payment_count THEN 1 ELSE 0 END) AS duplicate_sequence_orders,
       SUM(CASE WHEN min_sequence <> 1 THEN 1 ELSE 0 END) AS orders_not_starting_at_one,
       MAX(max_sequence) AS maximum_payment_sequence
FROM s
""").df())

display(con.sql("""
SELECT payment_type, COUNT(*) AS payment_records
FROM payments_clean
GROUP BY payment_type
ORDER BY payment_records DESC
""").df())

,payment_rows,unique_orders,payment_value_non_positive,installments_non_positive,sequence_non_positive
0,103886,99440,9.0,2.0,0.0


,orders_represented,duplicate_sequence_orders,orders_not_starting_at_one,maximum_payment_sequence
0,99440,0.0,80.0,29


,payment_type,payment_records
0,credit_card,76795
1,boleto,19784
2,voucher,5775
3,debit_card,1529
4,not_defined,3


## 5. Data Clean - Reviews

Review scores are converted to numeric values and retained at review level.

No review score is imputed. Multiple reviews for an order are summarized
later using the mean review score.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE reviews_clean AS
SELECT
    review_id, order_id,
    TRY_CAST(review_score AS INTEGER) AS review_score,
    review_comment_title, review_comment_message,
    TRY_CAST(review_creation_date AS TIMESTAMP) AS review_creation_date,
    TRY_CAST(review_answer_timestamp AS TIMESTAMP) AS review_answer_timestamp
FROM raw_reviews
""")

display(con.sql("""
SELECT
    COUNT(*) AS review_rows,
    COUNT(DISTINCT order_id) AS orders_represented,
    SUM(CASE WHEN review_score IS NULL THEN 1 ELSE 0 END) AS missing_scores,
    SUM(CASE WHEN review_score IS NOT NULL AND (review_score < 1 OR review_score > 5)
             THEN 1 ELSE 0 END) AS scores_outside_1_to_5,
    COUNT(DISTINCT review_id) AS unique_review_ids
FROM reviews_clean
""").df())

,review_rows,orders_represented,missing_scores,scores_outside_1_to_5,unique_review_ids
0,99224,98673,0.0,0.0,98410


## 6. Data Clean - Products and Customer Tables

Product metadata is retained without blanket imputation. Completeness
indicators are created and category names are enriched with the supplied
translation table.

Customer identifiers are retained without modification because
`customer_unique_id` is the identity used for repeat-purchase analysis.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE category_translation_clean AS
SELECT DISTINCT product_category_name, product_category_name_english
FROM raw_category_translation
""")

con.execute("""
CREATE OR REPLACE TABLE products_clean AS
WITH p AS (
    SELECT
        product_id, product_category_name,
        TRY_CAST(product_name_lenght AS DOUBLE) AS product_name_lenght,
        TRY_CAST(product_description_lenght AS DOUBLE) AS product_description_lenght,
        TRY_CAST(product_photos_qty AS DOUBLE) AS product_photos_qty,
        TRY_CAST(product_weight_g AS DOUBLE) AS product_weight_g,
        TRY_CAST(product_length_cm AS DOUBLE) AS product_length_cm,
        TRY_CAST(product_height_cm AS DOUBLE) AS product_height_cm,
        TRY_CAST(product_width_cm AS DOUBLE) AS product_width_cm
    FROM raw_products
)
SELECT
    p.*,
    COALESCE(ct.product_category_name_english, p.product_category_name)
        AS product_category_english,
    (
        (p.product_category_name IS NULL)::INTEGER +
        (p.product_name_lenght IS NULL)::INTEGER +
        (p.product_description_lenght IS NULL)::INTEGER +
        (p.product_photos_qty IS NULL)::INTEGER
    ) AS catalogue_attributes_missing_count,
    (
        (p.product_weight_g IS NULL)::INTEGER +
        (p.product_length_cm IS NULL)::INTEGER +
        (p.product_height_cm IS NULL)::INTEGER +
        (p.product_width_cm IS NULL)::INTEGER
    ) AS physical_attributes_missing_count,
    (
        p.product_category_name IS NOT NULL AND
        p.product_name_lenght IS NOT NULL AND
        p.product_description_lenght IS NOT NULL AND
        p.product_photos_qty IS NOT NULL
    ) AS catalogue_complete,
    (
        p.product_weight_g IS NOT NULL AND
        p.product_length_cm IS NOT NULL AND
        p.product_height_cm IS NOT NULL AND
        p.product_width_cm IS NOT NULL
    ) AS physical_attributes_complete
FROM p
LEFT JOIN category_translation_clean ct
    ON p.product_category_name = ct.product_category_name
""")

con.execute("""
CREATE OR REPLACE TABLE customers_clean AS
SELECT customer_id, customer_unique_id,
       customer_zip_code_prefix, customer_city, customer_state
FROM raw_customers
""")

con.execute("CREATE OR REPLACE TABLE sellers_source AS SELECT * FROM raw_sellers")
con.execute("CREATE OR REPLACE TABLE geolocation_source AS SELECT * FROM raw_geolocation")

display(con.sql("""
SELECT table_name
FROM duckdb_tables()
WHERE table_name IN (
    'orders_clean','order_items_clean','payments_clean','reviews_clean',
    'products_clean','customers_clean','category_translation_clean',
    'sellers_source','geolocation_source'
)
ORDER BY table_name
""").df())

,table_name
0,category_translation_clean
1,customers_clean
2,geolocation_source
3,order_items_clean
4,orders_clean
5,payments_clean
6,products_clean
7,reviews_clean
8,sellers_source


## 7. Build Order-Level Features

One-to-many tables are aggregated before being joined to orders.

This keeps the final order-level grain at one row per order and avoids
duplicating order values during joins.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE order_item_features AS
SELECT order_id,
       COUNT(*) AS item_count,
       SUM(price) AS total_price,
       SUM(freight_value) AS total_freight,
       AVG(price) AS avg_item_price,
       COUNT(DISTINCT product_id) AS unique_products,
       COUNT(DISTINCT seller_id) AS unique_sellers,
       SUM(freight_value) / NULLIF(SUM(price), 0) AS freight_to_price_ratio
FROM order_items_clean
GROUP BY order_id
""")

con.execute("""
CREATE OR REPLACE TABLE payment_features AS
SELECT order_id,
       COUNT(*) AS payment_count,
       SUM(payment_value) AS total_payment_value,
       MAX(payment_installments) AS max_installments,
       COUNT(DISTINCT payment_type) AS payment_type_count
FROM payments_clean
GROUP BY order_id
""")

con.execute("""
CREATE OR REPLACE TABLE review_features AS
SELECT order_id,
       COUNT(*) AS review_count,
       AVG(review_score) AS review_score_avg,
       MIN(review_score) AS review_score_min,
       MAX(review_score) AS review_score_max
FROM reviews_clean
GROUP BY order_id
""")

con.execute("""
CREATE OR REPLACE TABLE order_product_features AS
SELECT oi.order_id,
       COUNT(DISTINCT p.product_category_english) AS unique_product_categories,
       SUM(CASE WHEN NOT p.catalogue_complete THEN 1 ELSE 0 END)
           AS incomplete_catalogue_product_count,
       SUM(CASE WHEN NOT p.physical_attributes_complete THEN 1 ELSE 0 END)
           AS incomplete_physical_product_count
FROM order_items_clean oi
LEFT JOIN products_clean p ON oi.product_id = p.product_id
GROUP BY oi.order_id
""")

display(con.sql("""
SELECT 'order_item_features' AS table_name, COUNT(*) AS rows FROM order_item_features
UNION ALL SELECT 'payment_features', COUNT(*) FROM payment_features
UNION ALL SELECT 'review_features', COUNT(*) FROM review_features
UNION ALL SELECT 'order_product_features', COUNT(*) FROM order_product_features
""").df())

,table_name,rows
0,order_item_features,98666
1,payment_features,99440
2,review_features,98673
3,order_product_features,98666


## 8. Build the Order-Level Analytical Table

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE order_level_base AS
SELECT
    o.*,
    c.customer_unique_id,
    i.item_count, i.total_price, i.total_freight, i.avg_item_price,
    i.unique_products, i.unique_sellers, i.freight_to_price_ratio,
    pay.payment_count, pay.total_payment_value, pay.max_installments,
    pay.payment_type_count,
    r.review_count, r.review_score_avg, r.review_score_min, r.review_score_max,
    pf.unique_product_categories,
    pf.incomplete_catalogue_product_count,
    pf.incomplete_physical_product_count
FROM orders_clean o
LEFT JOIN customers_clean c ON o.customer_id = c.customer_id
LEFT JOIN order_item_features i ON o.order_id = i.order_id
LEFT JOIN payment_features pay ON o.order_id = pay.order_id
LEFT JOIN review_features r ON o.order_id = r.order_id
LEFT JOIN order_product_features pf ON o.order_id = pf.order_id
""")

display(con.sql("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT order_id) AS unique_orders,
       SUM(CASE WHEN customer_unique_id IS NULL THEN 1 ELSE 0 END)
           AS missing_customer_identity
FROM order_level_base
""").df())

,rows,unique_orders,missing_customer_identity
0,99441,99441,0.0


## 9. Build Customer Repeat-Purchase Dataset

`ROW_NUMBER()` creates the chronological order number for each
`customer_unique_id`. A customer is a repeat customer when the total number
of orders is greater than one.

The first order is used as the customer-level record.

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE order_level AS
WITH sequenced AS (
    SELECT *,
           ROW_NUMBER() OVER (
               PARTITION BY customer_unique_id
               ORDER BY order_purchase_timestamp, order_id
           ) AS customer_order_number,
           COUNT(*) OVER (
               PARTITION BY customer_unique_id
           ) AS customer_order_count
    FROM order_level_base
)
SELECT *,
       CASE WHEN customer_order_count > 1 THEN 1 ELSE 0 END
           AS is_repeat_customer
FROM sequenced
""")

con.execute("""
CREATE OR REPLACE TABLE first_order AS
SELECT * FROM order_level
WHERE customer_order_number = 1
""")

con.execute("""
CREATE OR REPLACE TABLE customer_level_analytical AS
SELECT * FROM first_order
WHERE order_delivered_customer_date IS NOT NULL
""")

display(con.sql("""
SELECT COUNT(*) AS customers,
       COUNT(DISTINCT customer_unique_id) AS unique_customers,
       SUM(is_repeat_customer) AS repeat_customers,
       100.0 * AVG(is_repeat_customer) AS repeat_rate_pct
FROM customer_level_analytical
""").df())

,customers,unique_customers,repeat_customers,repeat_rate_pct
0,93254,93254,2877.0,3.085122


## 10. Validation

In [ ]:
print("=== ORDER-LEVEL VALIDATION ===")
display(con.sql("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT order_id) AS unique_order_ids,
       COUNT(*) - COUNT(DISTINCT order_id) AS duplicate_order_ids
FROM order_level
""").df())

print("=== CUSTOMER-LEVEL VALIDATION ===")
display(con.sql("""
SELECT COUNT(*) AS rows,
       COUNT(DISTINCT customer_unique_id) AS unique_customer_identities,
       COUNT(*) - COUNT(DISTINCT customer_unique_id) AS duplicate_customer_identities
FROM customer_level_analytical
""").df())

print("=== CRITICAL FEATURE MISSINGNESS ===")
display(con.sql("""
SELECT
    SUM(CASE WHEN review_score_avg IS NULL THEN 1 ELSE 0 END) AS review_score_avg,
    SUM(CASE WHEN total_payment_value IS NULL THEN 1 ELSE 0 END) AS total_payment_value,
    SUM(CASE WHEN max_installments IS NULL THEN 1 ELSE 0 END) AS max_installments,
    SUM(CASE WHEN delivery_delay_days IS NULL THEN 1 ELSE 0 END) AS delivery_delay_days,
    SUM(CASE WHEN total_freight IS NULL THEN 1 ELSE 0 END) AS total_freight,
    SUM(CASE WHEN total_price IS NULL THEN 1 ELSE 0 END) AS total_price,
    SUM(CASE WHEN is_late IS NULL THEN 1 ELSE 0 END) AS is_late,
    SUM(CASE WHEN freight_to_price_ratio IS NULL THEN 1 ELSE 0 END) AS freight_to_price_ratio
FROM customer_level_analytical
""").df())

=== ORDER-LEVEL VALIDATION ===


,rows,unique_order_ids,duplicate_order_ids
0,99441,99441,0


=== CUSTOMER-LEVEL VALIDATION ===


,rows,unique_customer_identities,duplicate_customer_identities
0,93254,93254,0


=== CRITICAL FEATURE MISSINGNESS ===


,review_score_avg,total_payment_value,max_installments,delivery_delay_days,total_freight,total_price,is_late,freight_to_price_ratio
0,615.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0


## 10a. Exclusion-Bias Check — Customers Whose First Order Was Never Delivered

`customer_level_analytical` restricts to customers whose **first order has a
delivered date** (`order_delivered_customer_date IS NOT NULL`). This is a
necessary restriction for measuring delivery *lateness*, but it silently
excludes every customer whose first order was never delivered at all
(cancelled, lost, unavailable, etc.).

A non-delivered first order is arguably a more extreme negative experience
than a late one. If these excluded customers have a markedly different
repeat-purchase rate than the retained population, the H1 effect estimated
later in the project may understate (or overstate) the true relationship
between first-order experience and retention.

This check quantifies the excluded population and its repeat behaviour so
the choice can be reported as a disclosed, evidence-based limitation rather
than a silent one.


In [ ]:
# Compare the retained analytical population against the excluded
# population (customers whose FIRST order was never delivered).

exclusion_check = con.sql("""
WITH first_orders AS (
    SELECT *
    FROM order_level
    WHERE customer_order_number = 1
),
labelled AS (
    SELECT
        customer_unique_id,
        is_repeat_customer,
        CASE
            WHEN order_delivered_customer_date IS NOT NULL
                THEN 'Retained: first order delivered'
            ELSE 'Excluded: first order not delivered'
        END AS population_group
    FROM first_orders
)
SELECT
    population_group,
    COUNT(*) AS customers,
    SUM(is_repeat_customer) AS repeat_customers,
    100.0 * AVG(is_repeat_customer) AS repeat_rate_pct
FROM labelled
GROUP BY population_group
ORDER BY population_group
""").df()

display(exclusion_check)


,population_group,customers,repeat_customers,repeat_rate_pct
0,Excluded: first order not delivered,2842,120.0,4.222379
1,Retained: first order delivered,93254,2877.0,3.085122


In [ ]:
# Formal comparison: does repeat rate differ between the retained and
# excluded populations? Uses the same two-proportion z-test framework as
# the main H1/H2 diagnostic analysis (Notebook 04) for consistency.

import numpy as np
from statsmodels.stats.proportion import proportions_ztest

retained = exclusion_check[
    exclusion_check["population_group"] == "Retained: first order delivered"
].iloc[0]
excluded = exclusion_check[
    exclusion_check["population_group"] == "Excluded: first order not delivered"
].iloc[0]

count = np.array([
    int(excluded["repeat_customers"]),
    int(retained["repeat_customers"]),
])
nobs = np.array([
    int(excluded["customers"]),
    int(retained["customers"]),
])

z_stat, p_value = proportions_ztest(count, nobs)

print(
    f"Retained population repeat rate : {retained['repeat_rate_pct']:.2f}% "
    f"(n={int(retained['customers']):,})"
)
print(
    f"Excluded population repeat rate : {excluded['repeat_rate_pct']:.2f}% "
    f"(n={int(excluded['customers']):,})"
)
print(f"z = {z_stat:.3f}, p = {p_value:.6f}")

if p_value < 0.05:
    print(
        "\nThe excluded population's repeat rate differs significantly from "
        "the retained population. This confirms the exclusion is not neutral "
        "and should be reported as a limitation with a stated direction of "
        "likely bias in the final report."
    )
else:
    print(
        "\nNo significant difference was detected. The exclusion still "
        "warrants disclosure, but there is no evidence it biases the "
        "headline repeat-rate or H1 estimates."
    )


Retained population repeat rate : 3.09% (n=93,254)
Excluded population repeat rate : 4.22% (n=2,842)
z = 3.436, p = 0.000591

The excluded population's repeat rate differs significantly from the retained population. This confirms the exclusion is not neutral and should be reported as a limitation with a stated direction of likely bias in the final report.


### Exclusion-Bias Check Result

The excluded population (customers whose first order was never delivered, n=2,842) shows a repeat-purchase rate of 4.22%, compared to 3.09% for the retained population (n=93,254) used throughout the rest of this project. This difference is statistically significant (z = 3.436, p = 0.0006).

The direction is the opposite of what was hypothesized: excluded customers repeat more often, not less.


In [ ]:
print("=== REPEAT-PURCHASE VALIDATION ===")
display(con.sql("""
WITH c AS (
    SELECT customer_unique_id,
           COUNT(DISTINCT order_id) AS calculated_order_count,
           MAX(customer_order_count) AS stored_order_count,
           MAX(is_repeat_customer) AS stored_repeat_flag
    FROM order_level
    GROUP BY customer_unique_id
)
SELECT COUNT(*) AS customer_records,
       SUM(CASE WHEN calculated_order_count <> stored_order_count THEN 1 ELSE 0 END)
           AS order_count_mismatches,
       SUM(CASE WHEN stored_repeat_flag <>
                     CASE WHEN calculated_order_count > 1 THEN 1 ELSE 0 END
                THEN 1 ELSE 0 END)
           AS repeat_flag_mismatches,
       SUM(CASE WHEN calculated_order_count > 1 THEN 1 ELSE 0 END)
           AS repeat_customers
FROM c
""").df())

=== REPEAT-PURCHASE VALIDATION ===


,customer_records,order_count_mismatches,repeat_flag_mismatches,repeat_customers
0,96096,0.0,0.0,2997.0


## 11. Export CSV Copies

DuckDB remains the primary data store.

CSV files are exported only as convenience copies for inspection or use in
tools that do not connect to DuckDB.

In [ ]:
order_level = con.sql("SELECT * FROM order_level").df()
customer_analytical = con.sql("SELECT * FROM customer_level_analytical").df()

order_level_path = PROCESSED_DIR / "order_level_clean.csv"
customer_level_path = ANALYTICAL_DIR / "customer_level_analytical.csv"

order_level.to_csv(order_level_path, index=False)
customer_analytical.to_csv(customer_level_path, index=False)

print(f"Order-level CSV copy: {order_level_path}")
print(f"Customer-level CSV copy: {customer_level_path}")
print(f"DuckDB database: {DUCKDB_PATH}")

Order-level CSV copy: d:\Learning\Data Science\Capstone Project\Final_Project_Olist_BYOC_Capstone\02_Data\02_Processed\order_level_clean.csv
Customer-level CSV copy: d:\Learning\Data Science\Capstone Project\Final_Project_Olist_BYOC_Capstone\02_Data\03_Analytical\customer_level_analytical.csv
DuckDB database: d:\Learning\Data Science\Capstone Project\Final_Project_Olist_BYOC_Capstone\04_DuckDB\olist_capstone.duckdb


## 12. Final DuckDB Check

These tables are physically stored in the persistent DuckDB database.

In [ ]:
display(con.sql("""
SELECT table_name
FROM duckdb_tables()
WHERE schema_name = 'main'
ORDER BY table_name
""").df())

print("Final dataset sizes:")
print(f"Order-level: {order_level.shape[0]:,} rows × {order_level.shape[1]:,} columns")
print(f"Customer-level: {customer_analytical.shape[0]:,} rows × {customer_analytical.shape[1]:,} columns")

con.close()
print("DuckDB connection closed.")

,table_name
0,category_translation_clean
1,customer_level_analytical
2,customer_level_featured
3,customers_clean
4,diagnostic_findings
5,first_order
6,geolocation_source
7,order_item_features
8,order_items_clean
9,order_level


Final dataset sizes:
Order-level: 99,441 rows × 34 columns
Customer-level: 93,254 rows × 34 columns
DuckDB connection closed.
